In [15]:
import os
from anthropic import Anthropic

# Udemy Labs note: the previous cell configures ANTHROPIC_API_KEY for this session.
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY first."

BETAS = ["managed-agents-2026-04-01"]
MODEL = os.environ.get("MODEL", "claude-sonnet-5")  # course default; swap as models update
client = Anthropic()
print("SDK ready, model:", MODEL)

SDK ready, model: claude-sonnet-5


In [16]:
env = client.beta.environments.create(
    name="football-analysis",
    config={
        "type": "cloud",
        "packages": {
            # The data-science stack, pre-installed at build time.
            "pip": ["pandas==2.2.0", "numpy", "matplotlib","fastparquet"],
        },
        "networking": {"type": "unrestricted"},
    },
    betas=BETAS,
)
print(f"env.id = {env.id}")

env.id = env_01VrdKENwD3C2Z3QCzBHo9kY


In [17]:
import getpass
import os

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ").strip()

print("Configure your Google Docs Managed Agents vault for this notebook.")
existing_vault_id = os.environ.get("GOOGLE_DOCS_VAULT_ID", "").strip()
if existing_vault_id.startswith("sk-ant-") or (existing_vault_id and not existing_vault_id.startswith("vlt_")):
    print("Clearing invalid GOOGLE_DOCS_VAULT_ID. It must be a vault id starting with 'vlt_', not an API key.")
    os.environ.pop("GOOGLE_DOCS_VAULT_ID", None)

if not os.environ.get("GOOGLE_DOCS_VAULT_ID"):
    os.environ["GOOGLE_DOCS_VAULT_ID"] = input(
        "Google Docs vault ID from Claude Console (vlt_...): "
    ).strip()

# Optional advanced override only. In the normal course path, leave this unset;
# the next Google Docs cell reads the MCP URL from the vault credential.
# os.environ["GOOGLE_DOCS_MCP_URL"] = "https://your-google-docs-mcp.example.com/mcp"
print("ANTHROPIC_API_KEY configured for this notebook session.")
print("GOOGLE_DOCS_VAULT_ID configured for this notebook session:", os.environ["GOOGLE_DOCS_VAULT_ID"])

Configure your Google Docs Managed Agents vault for this notebook.
ANTHROPIC_API_KEY configured for this notebook session.
GOOGLE_DOCS_VAULT_ID configured for this notebook session: vlt_011CewZx7ACapEYBk37aKU75


In [18]:
FOOTBALL_API_HOST = "v3.football.api-sports.io"
FOOTBALL_SECRET_NAME = "FOOTBALL_API_KEY"


def football_credential_exists(vault_id: str) -> bool:
    credentials = list(client.beta.vaults.credentials.list(vault_id, betas=BETAS))
    return any(
        getattr(c.auth, "type", None) == "environment_variable"
        and getattr(c.auth, "secret_name", None) == FOOTBALL_SECRET_NAME
        for c in credentials
    )


# The vault only injects secrets that were actually stored as vault credentials -
# having the key in your local .env doesn't make it reachable by the agent.
if not football_credential_exists(GOOGLE_DOCS_VAULT_ID):
    football_key = getpass.getpass(
        f"No '{FOOTBALL_SECRET_NAME}' credential found in vault {GOOGLE_DOCS_VAULT_ID}. "
        "Enter your api-sports.io key to store there: "
    ).strip()
    client.beta.vaults.credentials.create(
        GOOGLE_DOCS_VAULT_ID,
        display_name="api-sports.io football API key",
        auth={
            "type": "environment_variable",
            "secret_name": FOOTBALL_SECRET_NAME,
            "secret_value": football_key,
            # Scope the secret to the one host it's needed for.
            "networking": {"type": "limited", "allowed_hosts": [FOOTBALL_API_HOST]},
        },
        betas=BETAS,
    )
    print(f"Created '{FOOTBALL_SECRET_NAME}' credential in vault {GOOGLE_DOCS_VAULT_ID}.")
else:
    print(f"'{FOOTBALL_SECRET_NAME}' credential already present in vault {GOOGLE_DOCS_VAULT_ID}.")

Created 'FOOTBALL_API_KEY' credential in vault vlt_011CewZx7ACapEYBk37aKU75.


In [19]:
RESEARCH_BRIEF_GDOCS_SYSTEM = """\
ROLE
You are a data engineer. Your job is to query this API endpoint: v3.football.api-sports.io to get the latest matchday stats for each player in a team that has had fixtures.

AUTHENTICATION
The api-sports.io key is available in your shell as the environment variable
FOOTBALL_API_KEY. Call the API with the bash tool via curl, e.g.:
  curl -s -H "x-apisports-key: $FOOTBALL_API_KEY" "https://v3.football.api-sports.io/fixtures?..."
Do not use web_fetch for this API - it cannot attach the required auth header.

CONSTRAINTS
- You only use the API to retrieve data 
- You do not provide any analysis on the data 
- You keep nulls and don't modify the data retrieved
- You have 7500 requests per day as an API limit

TOOLS
You have bash, write, and a "google_docs" MCP server.
Use bash+curl to call the football API, write to assemble the parquet file,
then call the google_docs MCP to upload it to Google Drive.

DELIVERABLE
End every session by creating ONE parquet file containing the stats retrived from the API. 
This file needs to be saved in my FootballStats folder. Also have the file saved in /mnt/session/outputs/. Report the URL of the created Google Doc in your final message.
"""
print(RESEARCH_BRIEF_GDOCS_SYSTEM)

ROLE
You are a data engineer. Your job is to query this API endpoint: v3.football.api-sports.io to get the latest matchday stats for each player in a team that has had fixtures.

AUTHENTICATION
The api-sports.io key is available in your shell as the environment variable
FOOTBALL_API_KEY. Call the API with the bash tool via curl, e.g.:
  curl -s -H "x-apisports-key: $FOOTBALL_API_KEY" "https://v3.football.api-sports.io/fixtures?..."
Do not use web_fetch for this API - it cannot attach the required auth header.

CONSTRAINTS
- You only use the API to retrieve data 
- You do not provide any analysis on the data 
- You keep nulls and don't modify the data retrieved
- You have 7500 requests per day as an API limit

TOOLS
You have bash, write, and a "google_docs" MCP server.
Use bash+curl to call the football API, write to assemble the parquet file,
then call the google_docs MCP to upload it to Google Drive.

DELIVERABLE
End every session by creating ONE parquet file containing the stats retr

In [20]:
from urllib.parse import urlparse

GOOGLE_DOCS_VAULT_ID = os.environ.get("GOOGLE_DOCS_VAULT_ID", "").strip()
GOOGLE_DOCS_MCP_URL = os.environ.get("GOOGLE_DOCS_MCP_URL", "").strip()


def validate_mcp_url(url: str) -> None:
    """Catch missing or placeholder URLs before the API returns a generic 400."""
    parsed = urlparse(url)
    if not url or "REPLACE-ME" in url or parsed.scheme != "https" or not parsed.netloc:
        raise RuntimeError(
            "Set GOOGLE_DOCS_MCP_URL to a valid https MCP endpoint, or use a "
            "GOOGLE_DOCS_VAULT_ID whose credential contains an MCP server URL."
        )


def validate_vault_id(vault_id: str) -> None:
    """Catch common copy/paste mistakes before sessions.create."""
    if not vault_id or vault_id.startswith("vlt_REPLACE"):
        raise RuntimeError("Set GOOGLE_DOCS_VAULT_ID to your Claude Managed Agents vault id.")
    if vault_id.startswith("sk-ant-"):
        raise RuntimeError(
            "GOOGLE_DOCS_VAULT_ID currently contains an Anthropic API key. "
            "Paste the Google Docs vault id from Claude Console instead; it "
            "should start with 'vlt_'."
        )
    if not vault_id.startswith("vlt_"):
        raise RuntimeError(f"GOOGLE_DOCS_VAULT_ID should start with 'vlt_'. Got: {vault_id!r}")


def google_docs_mcp_url_from_vault(vault_id: str) -> str:
    """Read the MCP URL from the first Google Docs MCP OAuth credential."""
    credentials = list(client.beta.vaults.credentials.list(vault_id, betas=BETAS))
    mcp_credentials = [
        credential for credential in credentials
        if getattr(credential.auth, "type", None) == "mcp_oauth"
    ]
    google_credentials = [
        credential for credential in mcp_credentials
        if "google" in (
            f"{credential.display_name or ''} "
            f"{getattr(credential.auth, 'mcp_server_url', '')}"
        ).lower()
    ]

    if len(google_credentials) == 1:
        return google_credentials[0].auth.mcp_server_url
    if len(mcp_credentials) == 1:
        return mcp_credentials[0].auth.mcp_server_url

    names = [
        f"{credential.display_name or credential.id}: "
        f"{getattr(credential.auth, 'mcp_server_url', '<no mcp url>')}"
        for credential in mcp_credentials
    ]
    raise RuntimeError(
        "Could not uniquely identify the Google Docs MCP credential in "
        f"vault {vault_id}. Set GOOGLE_DOCS_MCP_URL explicitly. "
        f"Found MCP credentials: {names or 'none'}"
    )


validate_vault_id(GOOGLE_DOCS_VAULT_ID)

if not GOOGLE_DOCS_MCP_URL:
    GOOGLE_DOCS_MCP_URL = google_docs_mcp_url_from_vault(GOOGLE_DOCS_VAULT_ID)

validate_mcp_url(GOOGLE_DOCS_MCP_URL)
print("Google Docs MCP URL:", GOOGLE_DOCS_MCP_URL)

mcp_server = {"type": "url", "name": "google_docs", "url": GOOGLE_DOCS_MCP_URL}

agent = client.beta.agents.create(
    name="Football Parquet Ingestor (Google Docs)",
    model=MODEL,
    system=RESEARCH_BRIEF_GDOCS_SYSTEM,
    mcp_servers=[mcp_server],
    tools=[
        # Built-in toolset: web_search, web_fetch, write, read, bash, ...
        {"type": "agent_toolset_20260401"},
        # Expose the Google Docs MCP tools to the agent. Auth comes from the vault.
        {"type": "mcp_toolset",
         "mcp_server_name": "google_docs",
         "default_config": {"permission_policy": {"type": "always_allow"}}},
    ],
    betas=BETAS,
)
print("agent.id =", agent.id)


Google Docs MCP URL: https://drivemcp.googleapis.com/mcp/v1
agent.id = agent_01GtE5vyoPKGrQN5jACKMBon


In [21]:
session = client.beta.sessions.create(
    agent=agent.id,
    environment_id=env.id,
    title="Generate a parquet file",
    vault_ids=[GOOGLE_DOCS_VAULT_ID],  # without this, MCP + env-var secrets never get injected
    betas=BETAS,
)
print(f"session.id = {session.id}")

session.id = sesn_01VYd8LQCxDi4hJEZcV8aNMK


In [ ]:
with client.beta.sessions.events.stream(session.id, betas=BETAS) as stream:
    client.beta.sessions.events.send(session.id, events=[{
        "type": "user.message",
        "content": [{
            "type": "text",
            "text": (
                "Query the API for all matchdays and create seperate parquets for each match day of the Premier League 2026/2027 Season"
                "Print the installed pandas, numpy, and matplotlib and fastparquet versions"
                "save the results to /mnt/session/outputs/England_PREM_2026_MATCHDAY_*.parquet. The wildcard specifies the matchday"
                "Also deliver the results to the FootballStats folder in my Google Drive, print the URL here"
            ),
        }],
    }], betas=BETAS)
    for event in stream:
        if event.type == "agent.message":
            for b in event.content:
                if b.type == "text":
                    print(b.text, end="", flush=True)
        elif event.type == "agent.tool_use":
            print(f"\n[tool: {event.name}]")
        elif event.type == "agent.mcp_tool_use":
            print(f"\n[mcp tool: {event.name}]")
        elif event.type == "session.error":
            print(f"\n[SESSION ERROR] {event}")
        elif event.type == "session.status_idle":
            print("\n--- session idle ---")
            break

In [ ]:
# `shared/cost_meter.py` doesn't exist in this project (leftover from a course scaffold) -
# `session.usage.list_cost` already carries this directly, no helper module needed.
updated = client.beta.sessions.retrieve(session.id, betas=BETAS)
print("list_cost:", updated.usage.list_cost)
print("active_seconds:", updated.usage.active_seconds)